# 01 — Exploratory data analysis: post-earnings volatility

Descriptive look at the modeling table (`data/processed/model_table.parquet`, primary sample) before the B0/B1 baselines.

**Holdout lock.** Only the **dev** split (2006–16) is used for exploration. **Validation** (2017–19) appears in exactly one place, the target-stability plot in section 6. Holdout (2021–25) and excluded_2020 are never read: the parquet read filters on `split`, and an assertion stops the notebook if any other split gets through.

No models, no tuning, no clipping of the data. Every figure is also saved to `notebooks/figures/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve().parent          # notebook runs from vol_pipeline/notebooks
sys.path.insert(0, str(ROOT))
import config as C

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(exist_ok=True)

# Palette: one accent for single-series charts, a second hue only where two series are compared
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1"
plt.rcParams.update({
    "figure.figsize": (9, 5), "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 14, "axes.titlepad": 26, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 11, "axes.labelcolor": INK2, "axes.edgecolor": MUTED, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "legend.frameon": False, "text.parse_math": False,
})

def finish(fig, name, subtitle=None):
    if subtitle:
        fig.axes[0].text(0, 1.012, subtitle, transform=fig.axes[0].transAxes, color=INK2, fontsize=10.5, va="bottom")
    fig.savefig(FIG_DIR / f"{name}.png", facecolor="white")
    plt.show()

TARGETS = ["v_post_3", "v_post", "v_post_15"]

## 1. Overview & sanity checks

Read only the dev and validation splits, then assert that nothing else was loaded.

In [ ]:
ALLOWED = {"dev", "validation"}
df = pd.read_parquet(C.OUT_MODEL, filters=[("split", "in", sorted(ALLOWED))])

# ---- holdout lock: stop here if any other split got through ----
assert set(df["split"].unique()) <= ALLOWED, f"HOLDOUT LOCK: unexpected splits {set(df['split'].unique()) - ALLOWED}"
yr = df["session_date"].dt.year
for s in ALLOWED:
    lo, hi = C.SPLITS[s]
    assert yr[df["split"] == s].between(lo, hi).all(), f"HOLDOUT LOCK: {s} rows outside {lo}-{hi}"
assert yr.max() <= 2019, "HOLDOUT LOCK: post-2019 row loaded"

dev = df[df["split"] == "dev"].copy()
val = df[df["split"] == "validation"].copy()   # used ONLY in section 6
print(f"dev {len(dev):,} rows | validation {len(val):,} rows (validation is reserved for section 6)")

In [ ]:
overview = pd.DataFrame({
    "rows": df.groupby("split").size(),
    "first session": df.groupby("split")["session_date"].min().dt.date,
    "last session": df.groupby("split")["session_date"].max().dt.date,
    "% with SUE": df.groupby("split")["has_sue"].mean().mul(100).round(1),
    "unique firms (PERMNO)": df.groupby("split")["permno"].nunique(),
}).reindex(["dev", "validation"])
display(overview)

In [ ]:
stats = dev[["v_pre", *TARGETS]].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
stats.insert(2, "median daily vol (raw)", np.exp(dev[["v_pre", *TARGETS]].median()).map("{:.2%}".format))
print("Dev, log realized volatility  v = ln(sqrt(mean(r^2)))")
display(stats.round(3))

In [ ]:
MODEL_COLS = ["v_pre", *TARGETS, "sue_price", "log_mktcap", "sector", "when", "n_analysts"]
nulls = dev[MODEL_COLS].isna().sum().rename("nulls in dev").to_frame()
display(nulls.T)
print("Total nulls in modeling columns (dev):", int(nulls.values.sum()))

**Sanity checks.** Every primary-sample row has a SUE (it is a condition of the primary sample). The modeling columns are complete except for **one dev row missing `log_mktcap`**; the baselines drop that row, which is why they train on 84,940 dev rows.

## 2. The target: 5-day post-earnings log realized volatility

`v_post = ln(sqrt(mean(r²)))` over the 5 sessions after the release. A value of −3.5 is a daily volatility of about e^−3.5 ≈ 3.0%.

In [ ]:
v = dev["v_post"]
med = v.median()
q001 = v.quantile(0.001)

fig, ax = plt.subplots()
ax.hist(v, bins=120, color=BLUE, edgecolor="white", linewidth=0.3)
ax.axvline(med, color=INK, lw=1.5)
ax.annotate(f"median {med:.2f}\n= {np.exp(med):.1%} daily vol", xy=(med, ax.get_ylim()[1] * 0.92),
            xytext=(14, 0), textcoords="offset points", color=INK, fontsize=10.5, va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=1.0))
ax.axvline(q001, color=MUTED, lw=1, ls="--")
ax.annotate(f"lowest 0.1%\n(< {np.exp(q001):.2%} daily vol)", xy=(q001, ax.get_ylim()[1] * 0.5),
            xytext=(-8, 0), textcoords="offset points", ha="right", color=INK2, fontsize=10)
ax.set_title("Post-earnings volatility is roughly log-normal, with a long low tail")
ax.set_xlabel("v_post  (log of 5-day realized daily volatility)")
ax.set_ylabel("Earnings events (dev, 2006–16)")
finish(fig, "01_target_hist", f"Dev split, n = {len(v):,}. Raw values, no clipping.")
print(f"Median v_post: {med:.3f} (log)  =  {np.exp(med):.2%} daily volatility")

### The extreme low tail

About 2.4% of dev events have a daily volatility below 0.67% (v_post < −5), and a few dozen sit below 0.1%. The data are kept raw here; whether to winsorize the target is an open modeling question, and this is the evidence for it.

The table below shows what these events are. They are **not** mainly small, illiquid stocks (see the market-cap comparison under the table). Several of the lowest tickers look like **acquisition targets with a pending deal**, whose price is pinned near the offer price, e.g. WebEx (WEBX, Cisco 2007), Chattem (CHTT, Sanofi 2010), McAfee (MFE, Intel 2010), International Rectifier (IRF, Infineon 2014). That reading comes from the tickers and dates, not from a merger flag in the data; it is worth checking with a deal database before deciding how to treat the tail.

In [ ]:
tail = pd.DataFrame({
    "threshold (log)": [-5, -6, -7],
    "daily vol below": [f"{np.exp(t):.2%}" for t in (-5, -6, -7)],
    "events": [(v < t).sum() for t in (-5, -6, -7)],
    "% of dev": [f"{(v < t).mean():.2%}" for t in (-5, -6, -7)],
})
display(tail)

cols = ["ticker", "session_date", "v_post", "v_pre", "log_mktcap", "n_analysts", "sector"]
lowest = dev.nsmallest(10, "v_post")[cols].assign(
    daily_vol=lambda d: np.exp(d["v_post"]).map("{:.3%}".format),
    mktcap_musd=lambda d: (np.exp(d["log_mktcap"]) / 1e3).round(0))
print("10 lowest v_post events in dev (market cap in $M)")
display(lowest.round(2))
print(f"Median market cap: lowest 1% of v_post ${np.exp(dev.loc[v < v.quantile(0.01), 'log_mktcap']).median()/1e3:,.0f}M "
      f"vs all dev ${np.exp(dev['log_mktcap']).median()/1e3:,.0f}M")

## 3. Why past volatility is the workhorse

The B0 baseline uses only `v_pre`, the realized volatility over the 20 sessions before the release. The scatter shows why that is a strong floor.

In [ ]:
r = dev[["v_pre", "v_post"]].corr().iloc[0, 1]

fig, ax = plt.subplots(figsize=(8, 6))
hb = ax.hexbin(dev["v_pre"], dev["v_post"], gridsize=70, cmap="Blues", mincnt=1, bins="log", linewidths=0)
lims = (-7, -0.5)
ax.plot(lims, lims, color=MUTED, lw=1, ls="--")
ax.annotate("v_post = v_pre", xy=(-1.5, -1.5), xytext=(-4, 6), textcoords="offset points", color=INK2, fontsize=10, ha="right")
b = dev.assign(bin=pd.qcut(dev["v_pre"], 20)).groupby("bin", observed=True)[["v_pre", "v_post"]].mean()
ax.plot(b["v_pre"], b["v_post"], color=ORANGE, lw=2, marker="o", ms=5, label="mean v_post per v_pre ventile")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_title(f"Past volatility predicts post-earnings volatility (r = {r:.2f})")
ax.set_xlabel("v_pre  (log realized vol, 20 sessions before the release)")
ax.set_ylabel("v_post  (log realized vol, 5 sessions after)")
ax.legend(loc="upper left")
cb = fig.colorbar(hb, ax=ax, pad=0.02); cb.set_label("events per cell (log scale)", color=INK2); cb.outline.set_visible(False)
cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}")); cb.ax.yaxis.set_minor_formatter(plt.NullFormatter())
finish(fig, "02_vpre_vs_vpost", f"Dev split, n = {len(dev):,}. Points sit above the diagonal: earnings weeks are more volatile than normal weeks.")

In [ ]:
corr = pd.DataFrame({
    "horizon (sessions)": [3, 5, 15],
    "corr(v_pre, target)": [dev[["v_pre", t]].corr().iloc[0, 1] for t in TARGETS],
}, index=TARGETS).round(3)
display(corr)

**Reading.** The correlation rises with the horizon. Over 3 days the earnings jump dominates; over 15 days the window is mostly ordinary trading, where the stock's usual volatility level takes over. The mean of `v_post` sits above the diagonal across the whole range: the earnings week adds volatility on top of the baseline, and the binned line is flatter than 45°, so very calm and very volatile stocks both regress towards the middle.

## 4. The surprise, measured right

`sue_price` is the earnings surprise scaled by price: (actual − consensus) / price. For a *volatility* target, the question is whether the **size** of the surprise matters, regardless of its sign.

Deciles are formed within dev only: `abs_sue` is ranked within dev and cut into 10 equal-count bins. Error bars are 95% confidence intervals of the mean.

In [ ]:
dev["abs_sue"] = dev["sue_price"].abs()

def decile_means(x):
    d = pd.qcut(x.rank(method="first"), 10, labels=range(1, 11))
    g = dev.groupby(d, observed=True)
    return pd.DataFrame({"mean": g["v_post"].mean(), "ci": 1.96 * g["v_post"].sem(),
                         "x_lo": g[x.name].min(), "x_hi": g[x.name].max(), "n": g.size()})

abs_dec = decile_means(dev["abs_sue"])

fig, ax = plt.subplots()
ax.errorbar(abs_dec.index, abs_dec["mean"], yerr=abs_dec["ci"], color=BLUE, lw=2, marker="o", ms=8,
            capsize=3, mec="white", mew=1.5)
ax.set_xticks(range(1, 11))
ax.set_title("Bigger surprises, bigger moves, and the effect accelerates in the top deciles")
ax.set_xlabel("Decile of |surprise|  (1 = smallest, 10 = largest; ranked within dev)")
ax.set_ylabel("Mean v_post (log 5-day realized vol)")
lo, hi = abs_dec["mean"].iloc[0], abs_dec["mean"].iloc[-1]
ax.annotate(f"+{hi - lo:.2f} log pts\n≈ {np.exp(hi - lo) - 1:.0%} higher vol", xy=(10, hi), xytext=(-12, -4),
            textcoords="offset points", ha="right", va="top", color=INK, fontsize=10.5)
finish(fig, "03_surprise_abs_deciles", f"Dev split, {abs_dec['n'].iloc[0]:,} events per decile. |surprise| = |sue_price|.")
display(abs_dec.assign(x_lo=abs_dec["x_lo"].map("{:.5f}".format), x_hi=abs_dec["x_hi"].map("{:.5f}".format)).round(3))

In [ ]:
sgn_dec = decile_means(dev["sue_price"])

fig, ax = plt.subplots()
ax.errorbar(sgn_dec.index, sgn_dec["mean"], yerr=sgn_dec["ci"], color=BLUE, lw=2, marker="o", ms=8,
            capsize=3, mec="white", mew=1.5)
ax.set_xticks(range(1, 11))
ax.set_xticklabels([f"{i}\n{'big miss' if i == 1 else 'big beat' if i == 10 else ''}" for i in range(1, 11)])
ax.set_ylim(*plt.gcf().axes[0].get_ylim())
ax.set_title("Direction barely matters: big misses and big beats both raise volatility")
ax.set_xlabel("Decile of signed surprise  (sue_price; ranked within dev)")
ax.set_ylabel("Mean v_post (log 5-day realized vol)")
finish(fig, "04_surprise_signed_deciles", "Dev split. A U-shape: a straight line through this cannot fit both tails, so a signed linear term comes out near zero.")
display(sgn_dec.assign(x_lo=sgn_dec["x_lo"].map("{:.5f}".format), x_hi=sgn_dec["x_hi"].map("{:.5f}".format)).round(3))

**What this shows.** Volatility rises with the *size* of the surprise, and the relationship is **nonlinear**: gentle across the lower and middle deciles and steepest in the top three (decile 10 is about 0.67 log points, roughly 2×, above decile 2). One exception: decile 1, which holds surprises of essentially zero (including exact zeros, i.e. actual = consensus), sits above decile 2; why is not examined here. Against the *signed* surprise the pattern is a U: both tails (big misses and big beats) are more volatile than in-line results. This is why a signed linear `sue_price` term came out near zero in the first B1 run, and why a rank of |surprise| works better than a clipped linear |surprise| (B1b vs B1a).

Caveat: this is unconditional. Volatile stocks also tend to produce larger surprises (price-scaled), so part of this gradient overlaps with `v_pre`; the B1b ablation is what measures the surprise's *incremental* contribution.

## 5. Structure: size, sector, timing

In [ ]:
g = dev.dropna(subset=["log_mktcap"]).assign(bin=lambda d: pd.qcut(d["log_mktcap"], 20)).groupby("bin", observed=True)
size = pd.DataFrame({"x": g["log_mktcap"].median(), "mean": g["v_post"].mean(), "ci": 1.96 * g["v_post"].sem()})

fig, ax = plt.subplots()
ax.errorbar(size["x"], size["mean"], yerr=size["ci"], color=BLUE, lw=2, marker="o", ms=7, capsize=2, mec="white", mew=1.2)
ticks = [50, 200, 1_000, 5_000, 25_000, 100_000]   # $M
ax.set_xticks(np.log(np.array(ticks) * 1e3))
ax.set_xticklabels([f"${t/1e3:,.0f}B" if t >= 1000 else f"${t}M" for t in ticks])
ax.set_title("Larger firms move less after earnings")
ax.set_xlim(right=np.log(150_000 * 1e3))
ax.set_xlabel("Market capitalization on the last trading day before the release (log scale)")
ax.set_ylabel("Mean v_post (log 5-day realized vol)")
finish(fig, "05_size", "Dev split, 20 equal-count bins (bin median on x), 95% CI. Flat from about $100M to $500M, then falls steadily.")

In [ ]:
sec = dev.groupby("sector")["v_post"].agg(["mean", "size"]).sort_values("mean")
labels = {s: s.split("_", 1)[1].replace("_", " / ").capitalize() for s in sec.index}

fig, ax = plt.subplots(figsize=(9, 5.5))
y = np.arange(len(sec))
ax.barh(y, sec["mean"] - (-4.0), left=-4.0, color=BLUE, height=0.7, edgecolor="white", linewidth=2)
ax.set_yticks(y, [f"{labels[s]}  (n={n:,})" for s, n in zip(sec.index, sec["size"])])
for yi, m in zip(y, sec["mean"]):
    ax.text(m + 0.01, yi, f"{m:.2f}", va="center", color=INK, fontsize=10)
ax.set_xlim(-4.0, sec["mean"].max() + 0.15)
ax.grid(axis="y", visible=False)
ax.set_title("Finance and utilities are the calmest sectors after earnings")
ax.set_xlabel("Mean v_post (log 5-day realized vol; axis starts at −4.0)")
finish(fig, "06_sector", "Dev split, SIC division. Agriculture and public administration rest on few events.")

In [ ]:
order = ["pre_open", "intraday", "after_close"]
wh = dev.groupby("when")["v_post"].agg(["mean", "size", "sem"]).reindex(order)

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(wh))
ax.bar(x, wh["mean"] - (-4.0), bottom=-4.0, color=BLUE, width=0.6, edgecolor="white", linewidth=2,
       yerr=1.96 * wh["sem"], error_kw={"ecolor": INK2, "capsize": 4, "lw": 1})
ax.set_xticks(x, [f"{w.replace('_', ' ')}\nn = {n:,}" for w, n in zip(wh.index, wh["size"])])
for xi, m in zip(x, wh["mean"]):
    ax.text(xi, -3.98, f"{m:.2f}", ha="center", va="bottom", color="white", fontsize=11, fontweight="bold")
ax.set_ylim(-4.0, wh["mean"].max() + 0.15)
ax.grid(axis="x", visible=False)
ax.set_title("After-close releases are more volatile than pre-open")
ax.set_ylabel("Mean v_post (axis starts at −4.0)")
finish(fig, "07_when", f"Dev split, 95% CI. Intraday has only {wh.loc['intraday', 'size']:,} events in the primary sample: treat it as unreliable.")

**Intraday is nearly absent** from the primary sample (281 dev events, about 0.3%). Most intraday 8-Ks follow a press release that came out earlier, and those events fail the session-agreement filter. Any estimate for intraday releases rests on a few hundred events.

## 6. Target stability over time

The one place validation appears. Mean `v_post` by year of the reaction session, dev and validation. Holdout and 2020 are not loaded and not shown.

In [ ]:
yr = (pd.concat([dev, val]).assign(year=lambda d: d["session_date"].dt.year)
      .groupby(["split", "year"])["v_post"].agg(["mean", "sem", "size"]).reset_index())

fig, ax = plt.subplots()
for split, color in [("dev", BLUE), ("validation", ORANGE)]:
    s = yr[yr["split"] == split]
    ax.errorbar(s["year"], s["mean"], yerr=1.96 * s["sem"], color=color, lw=2, marker="o", ms=7,
                capsize=2, mec="white", mew=1.2, label=f"{split} ({s['year'].min()}–{s['year'].max()})")
    last = s.iloc[-1]
    ax.annotate(split, xy=(last["year"], last["mean"]), xytext=(8, 0), textcoords="offset points",
                color=INK, fontsize=10.5, va="center")
dev_mean = dev["v_post"].mean()
ax.axhline(dev_mean, color=MUTED, lw=1, ls="--")
ax.annotate(f"dev mean {dev_mean:.2f}", xy=(2006, dev_mean), xytext=(0, 4), textcoords="offset points", color=INK2, fontsize=10)
ax.set_xticks(range(2006, 2020))
ax.set_xticklabels([str(y) if y % 2 == 0 else "" for y in range(2006, 2020)])
ax.set_xlim(2005.5, 2020.2)
ax.set_title("Post-earnings volatility tracks the market regime")
ax.set_xlabel("Year of the reaction session")
ax.set_ylabel("Mean v_post (log 5-day realized vol)")
ax.legend(loc="upper right")
finish(fig, "08_v_post_by_year", "Dev and validation only, 95% CI. Holdout (2021–25) and 2020 are locked and not shown.")
display(yr.pivot(index="year", columns="split", values="mean").round(3).T)

In [ ]:
by_year = pd.concat([dev, val]).groupby(pd.concat([dev, val])["session_date"].dt.year)["v_post"].mean()
print(f"dev mean {dev['v_post'].mean():.3f} | validation mean {val['v_post'].mean():.3f}")
print(f"highest year {by_year.idxmax()} ({by_year.max():.3f}) | lowest year {by_year.idxmin()} ({by_year.min():.3f})")

**Drift.** The yearly level moves with the market regime: the 2008–09 crisis is the clear peak (−3.14 in 2008) and 2013 the trough (−3.73), a swing of about 0.6 log points within dev. Validation sits inside the dev range (−3.61, −3.47, −3.46) and its overall mean equals dev's (−3.514 vs −3.515), so there is no level shift from dev to validation, though 2018–19 run above the dev average. The README already notes that the holdout period is more volatile (mean −3.31 vs −3.51 in dev); that is not examined here because the holdout is locked.

## Figures for the deck

All saved in `notebooks/figures/`: `01_target_hist`, `02_vpre_vs_vpost`, `03_surprise_abs_deciles`, `04_surprise_signed_deciles`, `05_size`, `06_sector`, `07_when`, `08_v_post_by_year`.